# 01 — Diagnostic et nettoyage des données brutes

Entrée : `data/brut/transactions/` (Parquet, un dossier par jour).
Sortie : `data/propre/transactions.parquet`.

1. Lecture des fichiers Parquet
2. Diagnostic : on mesure chaque problème
3. Corrections, une par une
4. Bilan avant / après

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("fraude-01-nettoyage")
         .master("spark://spark-master:7077")
         .config("spark.driver.host", "spark-master")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "2g")
         .config("spark.sql.shuffle.partitions", "18")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.eventLog.enabled", "true")
         .config("spark.eventLog.dir", "file:///tmp/spark-events")
         .getOrCreate())

DATA = "/opt/spark-data"
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 00:02:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Lecture
Spark lit tous les dossiers `date=...` d'un coup et recrée la colonne `date`.

In [2]:
brut = spark.read.parquet(DATA + "/brut/transactions")
brut.createOrReplaceTempView("brut")

nb_lignes_brut = brut.count()
print("Lignes :", nb_lignes_brut)
print("Fichiers :", len(brut.inputFiles()))
brut.printSchema()

[Stage 1:=======================>                                  (4 + 6) / 10]

Lignes : 6426188
Fichiers : 78
root
 |-- type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: string (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- transaction_time: string (nullable = true)
 |-- date: date (nullable = true)



Spark lit les 78 fichiers d'un coup. Le schéma confirme que `amount`, `isFraud` et `transaction_time` sont du texte : il faudra les convertir.

## 2. Diagnostic
### Valeurs manquantes par colonne

In [3]:
brut.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in brut.columns]).show(vertical=True)

[Stage 4:==================================>                       (6 + 4) / 10]

-RECORD 0------------------
 type             | 63993  
 amount           | 128347 
 nameOrig         | 0      
 oldbalanceOrg    | 192550 
 newbalanceOrig   | 0      
 nameDest         | 0      
 oldbalanceDest   | 128512 
 newbalanceDest   | 192659 
 isFraud          | 6468   
 isFlaggedFraud   | 0      
 transaction_time | 0      
 date             | 0      



On retrouve les proportions attendues : environ 2 % de montants manquants (128 347), 3 % pour `oldbalanceOrg` et `newbalanceDest`, 2 % pour `oldbalanceDest`, 1 % de types manquants (63 993) et 6 468 labels manquants.

### Lignes en double

In [4]:
print("Doublons :", nb_lignes_brut - brut.dropDuplicates().count())

[Stage 9:============================================>            (11 + 3) / 14]

Doublons : 63568


63 568 lignes en double, soit 1 % du total.

### Libellés de `type`

In [5]:
spark.sql("SELECT type, COUNT(*) AS nb FROM brut GROUP BY type ORDER BY nb DESC").show(20)

+----------+-------+
|      type|     nb|
+----------+-------+
|  CASH_OUT|1921378|
|   PAYMENT|1912402|
|   CASH_IN|1243525|
|  TRANSFER| 457246|
|  cash_out| 158373|
|   payment| 152124|
|   cash_in|  98868|
| CASH_OUT |  90275|
|  PAYMENT |  86881|
|   CASHOUT|  67442|
|      NULL|  63993|
|  CASH_IN |  56511|
|  transfer|  37823|
|     DEBIT|  36799|
| TRANSFER |  21660|
|       TRF|  16257|
|     debit|   2901|
|    DEBIT |   1730|
+----------+-------+



18 libellés différents au lieu de 5 : minuscules, espaces autour du mot, abréviations (`TRF`, `CASHOUT`) et valeurs manquantes.

### Valeurs du label `isFraud`

In [6]:
spark.sql("SELECT isFraud, COUNT(*) AS nb FROM brut GROUP BY isFraud ORDER BY nb DESC").show()

+-------+-------+
|isFraud|     nb|
+-------+-------+
|      0|6097023|
|     no| 314409|
|      1|   7916|
|   NULL|   6468|
|    yes|    372|
+-------+-------+



Le label mélange deux codages (`0`/`1` et `no`/`yes`), et 6 468 lignes n'en ont pas.

### Montants : texte avec virgule, négatifs, code d'erreur

In [7]:
spark.sql("""
    SELECT SUM(CASE WHEN amount LIKE '%,%' THEN 1 ELSE 0 END)                                 AS avec_virgule,
           SUM(CASE WHEN try_cast(REPLACE(amount, ',', '.') AS DOUBLE) < 0 THEN 1 ELSE 0 END)   AS negatifs,
           SUM(CASE WHEN try_cast(REPLACE(amount, ',', '.') AS DOUBLE) >= 9999999999 THEN 1 ELSE 0 END) AS code_erreur
    FROM brut
""").show()

+------------+--------+-----------+
|avec_virgule|negatifs|code_erreur|
+------------+--------+-----------+
|      314860|   31105|       6230|
+------------+--------+-----------+



314 860 montants écrits avec une virgule, 31 105 montants négatifs et 6 230 codes d'erreur.

### Formats de date

In [8]:
spark.sql("""
    SELECT SUM(CASE WHEN transaction_time LIKE '____-__-__ __:__:__' THEN 1 ELSE 0 END) AS format_yyyy_MM_dd,
           SUM(CASE WHEN transaction_time LIKE '__/__/____ __:__' THEN 1 ELSE 0 END)    AS format_dd_MM_yyyy
    FROM brut
""").show()

+-----------------+-----------------+
|format_yyyy_MM_dd|format_dd_MM_yyyy|
+-----------------+-----------------+
|          6105051|           321137|
+-----------------+-----------------+



321 137 dates (5 %) sont au format `dd/MM/yyyy HH:mm`. Les deux formats doivent devenir un seul type `timestamp`.

## 3. Corrections
### 3.1 Supprimer les doublons

In [9]:
propre = brut.dropDuplicates()

### 3.2 Label : `yes`/`1` → 1.0, `no`/`0` → 0.0, et suppression des lignes sans label
On ne peut pas apprendre sur une ligne dont on ne connaît pas la classe.

In [10]:
propre = propre.withColumn("label",
                           F.when(F.col("isFraud").isin("1", "yes"), 1.0)
                            .when(F.col("isFraud").isin("0", "no"), 0.0))
propre = propre.filter(F.col("label").isNotNull()).drop("isFraud")

### 3.3 Type : majuscules, espaces retirés, abréviations remplacées, lignes sans type supprimées

In [11]:
propre = propre.withColumn("type", F.upper(F.trim("type")))
propre = propre.withColumn("type",
                           F.when(F.col("type") == "TRF", "TRANSFER")
                            .when(F.col("type") == "CASHOUT", "CASH_OUT")
                            .otherwise(F.col("type")))
propre = propre.filter(F.col("type").isNotNull())

propre.groupBy("type").count().show()

[Stage 27:===================================================>    (12 + 1) / 13]

+--------+-------+
|    type|  count|
+--------+-------+
|   DEBIT|  40976|
| CASH_IN|1383815|
| PAYMENT|2127811|
|TRANSFER| 527206|
|CASH_OUT|2213096|
+--------+-------+



Il ne reste que les 5 types attendus.

### 3.4 Montant : texte → nombre, signe corrigé, puis code d'erreur → manquant

Le code 9999999999 n'est pas un vrai montant : on le remplace par une valeur manquante.
Les montants manquants seront remplacés par la médiane dans le pipeline (`Imputer`, notebook 03),
calculée **uniquement sur l'entraînement**.

In [12]:
# virgule -> point, puis texte -> nombre
propre = propre.withColumn("amount", F.regexp_replace("amount", ",", ".").try_cast("double"))
# erreur de signe
propre = propre.withColumn("amount", F.abs("amount"))
# code d'erreur -> valeur manquante (après abs, pour attraper aussi -9999999999)
propre = propre.withColumn("amount", F.when(F.col("amount") >= 9999999999, F.lit(None)).otherwise(F.col("amount")))

### 3.5 Date : on accepte les deux formats

In [13]:
# on essaie le premier format, sinon le second
propre = propre.withColumn("transaction_time",
                           F.coalesce(F.try_to_timestamp("transaction_time", F.lit("yyyy-MM-dd HH:mm:ss")),
                                      F.try_to_timestamp("transaction_time", F.lit("dd/MM/yyyy HH:mm"))))
propre = propre.drop("date")

### 3.6 Colonne inutile
`isFlaggedFraud` est une règle du simulateur, presque toujours à 0.

In [14]:
propre = propre.drop("isFlaggedFraud")

## 4. Écriture et bilan

In [15]:
propre.write.mode("overwrite").parquet(DATA + "/propre/transactions.parquet")

propre = spark.read.parquet(DATA + "/propre/transactions.parquet")
print("Lignes avant :", nb_lignes_brut, "| après :", propre.count())
propre.printSchema()

Lignes avant : 6426188 | après : 6292904
root
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- label: double (nullable = true)



On passe de 6 426 188 à 6 292 904 lignes : 63 568 doublons supprimés, puis les lignes sans label ou sans type. Les colonnes ont maintenant le bon type : `amount` et `label` en `double`, `transaction_time` en `timestamp`.

Valeurs manquantes restantes (traitées par l'`Imputer` dans le pipeline) :

In [16]:
propre.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in propre.columns]).show(vertical=True)

[Stage 38:===================>                                      (3 + 6) / 9]

-RECORD 0------------------
 type             | 0      
 amount           | 131752 
 nameOrig         | 0      
 oldbalanceOrg    | 188564 
 newbalanceOrig   | 0      
 nameDest         | 0      
 oldbalanceDest   | 125772 
 newbalanceDest   | 188699 
 transaction_time | 0      
 label            | 0      



Il reste des montants manquants (dont les anciens codes d'erreur) et des soldes manquants. On ne les remplace pas ici : l'`Imputer` du notebook 03 le fera avec la médiane calculée uniquement sur l'entraînement, pour ne rien apprendre du jeu de test.

In [17]:
spark.stop()